# Virtual Inertia Scheduling Demo

This notebook loads a trained model from the sweep outputs, predicts COI metrics, and
selects candidate (M, D) values that minimize COI deviation. The comparison uses
predictions and the observed labels from the dataset for the baseline row.

In [6]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

NOTEBOOK_DIR = Path("..").resolve()
TO_EXPORT = NOTEBOOK_DIR
sys.path.insert(0, str(TO_EXPORT))

from models.data_utils_local import load_dataset
from models.models_local import create_model

DATA_CSV = Path("../results/011626/run1/simulation_results.csv")
MODEL_DIR = Path("../outputs/to_export_sweep/MTLSH__huber__minmax__seed42__batch_eval128__batch_train64__epochs200__lr0.0005__use_lr_schedulerFalse__weight_decay1e-05")
MODEL_TYPE = "MTLSH"

assert DATA_CSV.exists(), f"Missing data file: {DATA_CSV}"
assert MODEL_DIR.exists(), f"Missing model dir: {MODEL_DIR}"

In [9]:
# Load dataset to get feature ordering
X, y, feature_cols, target_cols = load_dataset(DATA_CSV, target_cols=["rocof_max_COI","rocof_min_COI","devdown_COI","devup_COI","Delta_P_IBR_1","Delta_P_IBR_2","Delta_P_IBR_3","Delta_P_IBR_4"], remove_cols=["sim_id","seed","success","load_step_time","time_max_dev","plotter_csv"])
df = pd.read_csv(DATA_CSV)
df.shape

(3000, 67)

In [10]:
# Choose a baseline row with success == True
df_ok = df[df["success"] == True].copy()
assert len(df_ok) > 0, "No successful rows found."
row = df_ok.iloc[0]
row[target_cols]

rocof_max_COI    0.214036
rocof_min_COI   -0.016434
devdown_COI           0.0
devup_COI        0.393781
Delta_P_IBR_1   -1.037385
Delta_P_IBR_2   -0.858845
Delta_P_IBR_3   -0.581166
Delta_P_IBR_4   -1.856771
Name: 0, dtype: object

In [12]:
# Load scalers and model
import joblib
x_scaler = joblib.load(MODEL_DIR / "x_scaler.pkl")
y_scaler = joblib.load(MODEL_DIR / "y_scaler.pkl")

model, device = create_model(MODEL_TYPE, in_dim=len(feature_cols), out_dim=len(target_cols))
state = torch.load(MODEL_DIR / "vis_mlp_state_dict.pt", map_location=device)
model.load_state_dict(state)
model.eval()

MTLSharedHeads(
  (shared): Sequential(
    (0): Linear(in_features=53, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.0, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.0, inplace=False)
  )
  (heads): ModuleList(
    (0-7): 8 x Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.0, inplace=False)
      (3): Linear(in_features=128, out_features=64, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.0, inplace=False)
      (6): Linear(in_features=64, out_features=1, bias=True)
    )
  )
)

In [13]:
def predict_from_features(feat_vec):
    feat_vec = np.asarray(feat_vec, dtype=np.float32).reshape(1, -1)
    feat_norm = x_scaler.transform(feat_vec)
    with torch.no_grad():
        pred_norm = model(torch.from_numpy(feat_norm).to(device)).cpu().numpy()
    pred = y_scaler.inverse_transform(pred_norm).reshape(-1)
    return {k: float(v) for k, v in zip(target_cols, pred)}

def build_feature_vector(row_dict):
    return [row_dict.get(col, 0.0) for col in feature_cols]

In [14]:
# Baseline prediction vs true
base_feat = build_feature_vector(row)
base_pred = predict_from_features(base_feat)
base_true = row[target_cols].to_dict()

print("Baseline prediction vs true (subset):")
for name in ["devdown_COI", "devup_COI", "rocof_max_COI", "rocof_min_COI"]:
    if name in base_pred:
        print(f"{name}: pred={base_pred[name]:.4f} | true={base_true.get(name, np.nan):.4f}")

Baseline prediction vs true (subset):
devdown_COI: pred=-0.0011 | true=0.0000
devup_COI: pred=0.3981 | true=0.3938
rocof_max_COI: pred=0.2201 | true=0.2140
rocof_min_COI: pred=-0.0149 | true=-0.0164


In [16]:
# Candidate scheduling using model predictions
rng = np.random.default_rng(42)
n_candidates = 128

m_cols = [c for c in feature_cols if c.startswith("M_") and c[2:].isdigit()]
d_cols = [c for c in feature_cols if c.startswith("D_") and c[2:].isdigit()]

m_low, m_high = 2.0, 6.0
d_low, d_high = 0.0, 3.0

best = {"score": float("inf"), "pred": None, "row": None}

for _ in range(n_candidates):
    cand = row.copy()
    m_vals = rng.uniform(m_low, m_high, size=len(m_cols))
    d_vals = rng.uniform(d_low, d_high, size=len(d_cols))
    for col, val in zip(m_cols, m_vals):
        cand[col] = float(val)
    for col, val in zip(d_cols, d_vals):
        cand[col] = float(val)

    pred = predict_from_features(build_feature_vector(cand))
    score = abs(pred.get("devdown_COI", 0.0)) + abs(pred.get("devup_COI", 0.0))
    if score < best["score"]:
        best = {"score": score, "pred": pred, "row": cand}

best


{'score': 0.39131314028054476,
 'pred': {'rocof_max_COI': 0.21525608003139496,
  'rocof_min_COI': -0.018405752256512642,
  'devdown_COI': -0.0016047703102231026,
  'devup_COI': 0.38970836997032166,
  'Delta_P_IBR_1': -0.907302737236023,
  'Delta_P_IBR_2': -0.832378089427948,
  'Delta_P_IBR_3': -0.7226031422615051,
  'Delta_P_IBR_4': -0.880643904209137},
 'row': sim_id                                                             0
 seed                                                              42
 success                                                         True
 base_load_scale                                              0.99656
 load_step_scale                                             0.863327
                                          ...                        
 Delta_P_IBR_1                                              -1.037385
 Delta_P_IBR_2                                              -0.858845
 Delta_P_IBR_3                                              -0.581166
 Delta_

## Comparison
The model-selected candidate below is the best according to predicted COI deviations.
Ground truth for this candidate is not available unless you re-run a simulation with the
selected M/D values. The baseline row comparison uses actual labels from the dataset.

In [19]:
print("Baseline devdown/devup: ", base_true.get("devdown_COI"), base_true.get("devup_COI"))
print("Scheduled (pred) devdown/devup: ", best["pred"].get("devdown_COI"), best["pred"].get("devup_COI"))

Baseline devdown/devup:  0.0 0.3937806104063455
Scheduled (pred) devdown/devup:  -0.0016047703102231026 0.38970836997032166
